# SD image benchmark on Kaggle
### Checkpoints, reference methods, queue and portable resume

- **Created**: 2026-11-09T14:20:00+07:00
- **Last Updated**: 2026-11-09T14:20:00+07:00
- **Author**: Group 4
- **Objective**: Run real image inference in small batches, retain failures and resume validated outputs. GPU compatibility of the full catalog remains experimental until actually run.

## References & Consumed Artifacts

- **Source Code**: [queue](../src/pipelines/sd_all_benchmark.py), [archive](../src/pipelines/sd_benchmark_archive.py), [data preparation](../src/pipelines/prepare_image_demo.py), [preview](../src/common/visualizer.py)
- **Checkpoints / Results**: [run directory](../experiments/runs/)
- **Configuration**: [suite](../configs/generation/sd_all_benchmark.yaml), [pilot](../configs/generation/kaggle_image_demo.yaml)
- **Guide**: [benchmark guide](../docs/reports/KAGGLE_SD_ALL_BENCHMARK.md)
- **Governing Rule**: [notebook separation](../.agents/rules/LOGGING_CHECKPOINT_RULES.md)

Enable GPU and Internet. Attach `SD_ALL_BENCHMARK_KAGGLE.zip` as a Kaggle Dataset (ZIP or extracted code). Use images you are authorized to process. Some public weights are research-only or gated; free download does not imply unrestricted licensing. Large SD3.5 models may exceed available memory/disk.


In [ ]:
from pathlib import Path
import os
import sys
import zipfile

workspace = Path.cwd()
if (workspace / "src/pipelines/sd_all_benchmark.py").is_file():
    repo = workspace
elif (workspace / "uth-cv-coursework/src/pipelines/sd_all_benchmark.py").is_file():
    repo = workspace / "uth-cv-coursework"
else:
    import shutil
    input_dir = workspace.parent / "input"
    archives = sorted(input_dir.rglob("SD_ALL_BENCHMARK_KAGGLE.zip"))
    unpacked = [p.parents[2] for p in input_dir.rglob("sd_all_benchmark.py")
                if p.parent.name == "pipelines" and p.parent.parent.name == "src"
                and (p.parents[2] / "configs/generation/kaggle_image_demo.yaml").is_file()]
    repo = workspace / "uth-cv-coursework"
    if unpacked:
        shutil.copytree(unpacked[0], repo, dirs_exist_ok=True)
    elif archives:
        with zipfile.ZipFile(archives[0]) as bundle:
            for member in bundle.infolist():
                if not (repo / member.filename).resolve().is_relative_to(repo.resolve()):
                    raise ValueError("Unsafe archive path")
            bundle.extractall(repo)
    else:
        raise FileNotFoundError("Attach SD_ALL_BENCHMARK_KAGGLE.zip or its extracted code Dataset first.")
os.chdir(repo)
sys.path.insert(0, str(repo))
repo


Inference dependencies are installed inside worker venvs. The notebook kernel installs only requirements/notebook.txt. If an older cell already downgraded Hub/protobuf globally, save results and start a fresh Kaggle session before using this notebook; restarting only the Python kernel does not undo pip changes.

In [ ]:
%pip install -q -r requirements/notebook.txt


In [ ]:
import json
import html
import torch
import yaml
from IPython.display import HTML, Image, FileLink, display
from src.pipelines.prepare_image_demo import prepare_drive_demo
from src.common.visualizer import save_sd_input_preview
from src.pipelines.sd_all_benchmark import run_suite
from src.pipelines.sd_benchmark_archive import restore_results, export_results

assert torch.cuda.is_available(), "Enable Kaggle GPU before running inference."
config_path = Path("configs/generation/sd_all_benchmark.yaml")
suite = yaml.safe_load(config_path.read_text(encoding="utf-8"))
pilot = yaml.safe_load(Path(suite["pilot_config"]).read_text(encoding="utf-8"))
run_dir = Path(suite["run_dir"])
display({"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__})


## Optional: restore a previous session first
Set `RESTORE_ZIP` to an attached `sd_benchmark_results.zip`, or leave `None` for a new run. Restore requires an empty run directory. Prepare the same original inputs in the next cell and keep this source bundle version; changed code/config/input hashes intentionally schedule new jobs. Kaggle working storage alone does not survive a fresh session.

For gated checkpoints, accept their model terms and provide your Hugging Face token through Kaggle Secrets/environment `HF_TOKEN`; never paste it into saved notebook cells or outputs.


In [ ]:
RESTORE_ZIP = None  # e.g. Path("/kaggle/input/previous-run/sd_benchmark_results.zip")
if RESTORE_ZIP is not None:
    restore_results(Path(RESTORE_ZIP), run_dir)


## Prepare and inspect three real images
The pilot downloads only selected adult FFHQ crops. For your own inputs, supply a JSONL manifest as described in the guide. Review the mouth mask before inpainting; it is a pilot ellipse, not automatic face alignment.


In [ ]:
data = pilot["data"]
selected = prepare_drive_demo(data["drive_folder_id"], Path(data["input_manifest"]), Path(data["download_root"]), limit=data["limit"])
display([{k: r[k] for k in ("image_id", "file_path", "split")} for r in selected])
preview = Path(pilot["run_dir"]) / "input_preview.png"
generation = pilot["generation"]
save_sd_input_preview(Path(data["input_manifest"]), preview, (generation["width"], generation["height"]), tuple(generation["mask_box_xyxy_normalized"]))
display(Image(filename=str(preview)))


In [ ]:
state = run_suite(config_path, plan_only=True)
display({"catalog_entries": state["catalog_entries"], "generation_cases": len(state["jobs"]),
         "framework_tools": [f["name"] for f in state["frameworks"]]})
display(HTML((run_dir / "index.html").read_text(encoding="utf-8")))


## Run a small batch; rerun this cell to advance the queue
`GROUP`: `all`, `checkpoints`, or `methods`. `CASE_IDS=None` selects the group; a list such as `["sd15", "sd15_inpaint"]` selects specific cases. `MAX_JOBS` counts attempts, including failed jobs. Failed jobs remain recorded and are skipped until `RETRY_FAILED=True`. No mock fallback occurs.

The 32 generation configurations expand to 44 mode tasks per source (18 checkpoint configurations + 14 reference configurations); these are not 32 independent research methods. The 3 framework tools do not generate additional benchmark samples. Reference generation and face swapping are separate tasks and must not be ranked together as if equivalent.


In [ ]:
GROUP = "all"
CASE_IDS = None
MAX_JOBS = 2
RETRY_FAILED = False
state = run_suite(config_path, group=GROUP, case_ids=CASE_IDS, max_jobs=MAX_JOBS,
                  retry_failed=RETRY_FAILED, keep_model_cache=False)
display(HTML((run_dir / "index.html").read_text(encoding="utf-8")))


## Inspect actual results and failures
Each case directory contains `worker.log`, resolved configuration/provenance and, on success, a self-contained `gallery.html`, generated manifest and technical QA. `completed` means the expected images decoded and matched hashes. Identity fidelity, realism, demographic coverage and detector usefulness are not evaluated here. Native sizes and method-specific prompts are retained in metadata.

Optional upstream dependencies can fail to install on the current Kaggle runtime; these cases remain failures, not low-quality scores. Read the logs before retrying. Per-case model caches and upstream scratch files are removed by default after execution; the venv cache remains within the run directory. One GPU job runs at a time.


In [ ]:
completed = [j for j in state["jobs"] if j["status"] == "completed"]
if completed:
    gallery_path = Path(completed[-1]["directory"]) / "gallery.html"
    markup = html.escape(gallery_path.read_text(encoding="utf-8"), quote=True)
    display(HTML(f'<iframe srcdoc="{markup}" style="width:100%;height:850px;border:0"></iframe>'))
else:
    display("No completed image case yet. Inspect case worker.log files.")


In [ ]:
archive = export_results(run_dir, run_dir.parent / "sd_benchmark_results.zip")
display(FileLink(str(archive)))


Download the ZIP or save this notebook's output before the session ends. Attach it to the next session and use the restore cell. Environments and weights are excluded and will be downloaded again. The original source images must still be available. Keep failure logs in the report; a failed/OOM/gated case is not evidence of poor image quality.
